# Getting Started with OpenData

This notebook walks through loading, exploring, filtering, and charting public datasets using the [OpenData Python SDK](https://pypi.org/project/tryopendata/).

No API key required. All public datasets are free to query.

**Hit Runtime → Run all** to run every cell at once, or step through them one by one.

In [ ]:
# Install the SDK with pandas support
!pip install -q tryopendata[pandas] matplotlib

## 1. Connect and inspect a dataset

Let's look at the US unemployment rate from the Federal Reserve. We'll check what columns it has and how many rows before loading any data.

In [ ]:
from opendata_sdk import OpenData, Query

client = OpenData()

# Inspect the dataset
meta = client.meta("fred/unemployment-rate")
print(f"Dataset:  {meta.name}")
print(f"Rows:     {meta.rows:,}")
print(f"Columns:  {[c.name for c in meta.columns]}")

## 2. Load data into a DataFrame

Load the full dataset and preview it. The SDK handles pagination automatically, so `load()` fetches all rows even if the dataset spans multiple pages.

In [ ]:
df = client.load("fred/unemployment-rate").to_pandas()
print(f"{len(df)} rows loaded")
df.head(10)

## 3. Filter and sort server-side

Use `Query()` to apply filters before the data is downloaded. Only matching rows get sent over the network.

In [ ]:
q = (
    Query()
    .gte("date", "2010-01-01")              # 2010 onward
    .fields("date", "unemployment_rate")     # just these columns
    .sort("date")                            # oldest first
)

df_filtered = client.load("fred/unemployment-rate", q).to_pandas()
print(f"Loaded {len(df_filtered)} rows")
df_filtered.head()

## 4. Plot a chart

pandas has built-in plotting via matplotlib. Let's chart unemployment over time. You should see the massive COVID-19 spike in April 2020.

In [ ]:
ax = df_filtered.assign(date=df_filtered["date"].astype(str)).plot(
    x="date",
    y="unemployment_rate",
    title="US Unemployment Rate (2010 - Present)",
    xlabel="Date",
    ylabel="Unemployment Rate (%)",
    figsize=(12, 5),
    legend=False,
)
ax.figure.tight_layout()

## 5. Search for more datasets

Find datasets by keyword, right from the notebook.

In [ ]:
results = client.search("consumer price index")
for ds in results.results:
    print(f"{ds.provider}/{ds.slug}: {ds.name}")
    print(f"  {(ds.description or '')[:100]}...")
    print()

## 6. Summary stats

Use standard pandas methods to explore your data.

In [ ]:
print(f"Unemployment range: {df_filtered['unemployment_rate'].min():.1f}% to {df_filtered['unemployment_rate'].max():.1f}%")
print(f"Latest: {df_filtered['unemployment_rate'].iloc[-1]:.1f}%")
print()
df_filtered.describe()

## Next steps

- Browse datasets: [tryopendata.ai/explore](https://tryopendata.ai/explore)
- Jupyter Notebooks guide: [tryopendata.ai/docs/guides/jupyter](https://tryopendata.ai/docs/guides/jupyter)
- Python SDK docs: [tryopendata.ai/docs/guides/python-sdk](https://tryopendata.ai/docs/guides/python-sdk)
- Querying guide: [tryopendata.ai/docs/guides/querying](https://tryopendata.ai/docs/guides/querying)
- GitHub: [github.com/tryopendata/opendata-python](https://github.com/tryopendata/opendata-python)